# Photonic time crystal and time reflection

*waves · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/photonic_time_crystal)

Light in a medium whose permittivity $\varepsilon(t)$ changes in time but not in space. A sudden jump of $\varepsilon$ splits a pulse into a forward and a time-reflected backward wave at a new frequency; a periodic modulation opens a band gap in momentum $k$, inside which waves grow exponentially. Every Fourier mode is an independent $2\times 2$ problem, so all $k$ are solved as one batch, and the Floquet bands come from one batched eigvals call.

**Vector form**

$$\partial_t D = -\partial_x H,\quad \partial_t B = -\partial_x E,\quad D = \varepsilon(t)\,E,\ B = H \qquad \ddot D_k + \frac{k^2}{\varepsilon(t)}\,D_k = 0$$

**Index (tensor) form**

$$\dot u_{ka} = A_{kab}(t)\,u_{kb},\quad A_k = \begin{pmatrix} 0 & -ik \\ -ik/\varepsilon(t) & 0 \end{pmatrix} \qquad M_{kab} = \mathcal T\exp\!\int_0^T A_k\,dt,\quad M_{kab}\,w_{kb} = \lambda_k\,w_{ka},\quad \lambda_k = e^{-i\omega_k T}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{ka} = (D_k, B_k)$ | `stack([fft(D), fft(B)], dim=-1)` | (K, 2) | Fourier amplitudes; the medium is uniform in $x$, so $k$ is conserved and every mode evolves on its own |
| $A_{kab}(t)$ | `generator(k, eps)` | (K, 2, 2) | one $2\times 2$ generator per wavenumber |
| $A_{kab}\,u_{kb}$ | `einsum('kab,kb...->ka...', A, u)` | (K, 2) | repeated $b$ ⇒ summed; the same line evolves the monodromy matrix $(K, 2, 2)$ |
| $D_f,\ D_b$ | `temporal_fresnel(n1, n2)` | () | forward/backward amplitudes $\tfrac12(1 \pm n_2/n_1)$ from continuity of $D$ and $B$ |
| $M_{kab}$ | `monodromy(k_band, eps_periodic, T)` | (K, 2, 2) | propagator over one period, from RK4 applied to the identity matrix |
| $\lambda_k$ | `torch.linalg.eigvals(M)` | (K, 2) | Floquet multipliers for all $k$ in one batched call; $\det M = 1$, so $\lambda_+\lambda_- = 1$ |
| $\omega_k = i\ln\lambda_k / T$ | `floquet_frequencies(lam, T)` | (K, 2) | quasi-frequency: real in a band, $\operatorname{Im}\omega > 0$ (growth) in a momentum gap |

Swapping the roles of space and time turns familiar optics inside out. At an ordinary interface in space the frequency is conserved and the wavenumber changes; at a temporal interface, where the whole medium switches from $\varepsilon_1$ to $\varepsilon_2$ at once, the wavenumber $k$ is conserved and the frequency jumps, $\omega_2 = \omega_1 n_1/n_2$. Because no charges or currents appear, $D$ and $B$ stay continuous (not $E$ and $H$), which gives the temporal Fresnel coefficients $D_f = \tfrac12(1 + n_2/n_1)$ and $D_b = \tfrac12(1 - n_2/n_1)$ (Morgenthaler, 1958): part of the pulse turns around and runs backwards, the time reflection, and the energy is not conserved because switching the medium does work on the field. Temporal reflection and broadband frequency translation were observed with microwaves in a transmission-line metamaterial switched in time (Moussa et al., Nature Physics 2023). A periodic modulation $\varepsilon(t) = \varepsilon_0(1 + \alpha\cos\Omega t)$ makes each mode a parametric oscillator, $\ddot D_k + (k^2/\varepsilon)D_k = 0$ (a Mathieu-like equation). Near $k/n_0 = \Omega/2$ the forward and backward waves are coupled resonantly: the quasi-frequency locks to $\Omega/2$ and acquires an imaginary part, a band gap in momentum (rather than in frequency), with edges $k \approx \tfrac{n_0\Omega}{2}(1 \mp \alpha/4)$ and peak growth rate $\approx \alpha\Omega/8$ to first order in $\alpha$. A second, much narrower gap at $k \approx n_0\Omega$ appears at order $\alpha^2$. In the animation only the in-gap part of the pulse spectrum grows. This is the idea behind photonic time crystals (reviewed by Galiffi et al., "Photonics of time-varying media", Advanced Photonics 2022); at optical frequencies the modulation depth and speed needed are very hard to reach, so experiments so far work mainly at microwave and radio frequencies. Simplifications: a lossless, dispersionless medium in 1D, $\mu = 1$, an instantaneous jump, and a modulation that is imposed from outside (the energy for the growth comes from the pump).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/photonic_time_crystal/main.py
"""Main loop: a temporal interface (time reflection) and a periodic modulation (momentum gap), plus Floquet bands."""

import math
from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import equation, plot, solve, tensors

EPS0 = 1.0  # unmodulated permittivity (vacuum-like, so c = 1 before the jump)
DT = 0.05


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    alpha, omega, eps2 = params["alpha"], params["omega"], params["eps2"]
    t_max = 12.0 if fast_mode() else params["t_max"]
    t_jump = min(25.0, t_max / 3)
    every = max(1, round(t_max / DT / 200))  # keep about 200 snapshots
    x, k = tensors.grid(dev)
    k_gap = math.sqrt(EPS0) * omega / 2  # centre of the first momentum gap

    # 1. Temporal interface: ε jumps from EPS0 to eps2 at t_jump.
    n1, n2 = math.sqrt(EPS0), math.sqrt(eps2)
    D0, B0 = tensors.forward_pulse(x, k_gap, n1)
    times, u_jump = solve.interface(D0, B0, k, EPS0, eps2, t_jump, t_max, DT, every)
    D_jump = solve.to_field(u_jump)  # (S, X)
    T_f, T_b = equation.temporal_fresnel(n1, n2)
    D_fwd, D_bwd = equation.split_directions(D_jump[-1], solve.to_field(u_jump[-1], component=1), n2)
    print(f"temporal interface ε: {EPS0} → {eps2} at t = {t_jump:g}:  ω2/ω1 = n1/n2 = {n1 / n2:.3f}")
    print(f"  forward  D amplitude: Fresnel {T_f:+.3f}   simulated |D| {solve.envelope(D_fwd, k).max().item():.3f}")
    print(f"  backward D amplitude: Fresnel {T_b:+.3f}   simulated |D| {solve.envelope(D_bwd, k).max().item():.3f}")

    # 2. Periodic modulation ε(t) = EPS0 (1 + α cos Ωt), same pulse.
    def eps_periodic(t: float) -> float:
        return equation.periodic_eps(t, EPS0, alpha, omega)

    D0p, B0p = tensors.forward_pulse(x, k_gap, math.sqrt(eps_periodic(0.0)))
    times_p, u_per = solve.evolve(solve.to_modes(D0p, B0p), k, eps_periodic, 0.0, DT, round(t_max / DT), every)
    times_p = torch.tensor(times_p)
    D_per = solve.to_field(u_per)  # (S, X)
    spectra = u_per[..., 0].abs() / len(x)  # |D_k(t)|  (S, K)

    # 3. Floquet band structure: one monodromy matrix per k, eigenvalues in one batched call.
    period = 2 * math.pi / omega
    k_band = tensors.band_wavenumbers(2.4 * k_gap, 60 if fast_mode() else 480, dev)
    M = solve.monodromy(k_band, eps_periodic, period)  # (K, 2, 2)
    w_F = equation.floquet_frequencies(torch.linalg.eigvals(M), period)  # (K, 2)
    growth = w_F.imag.max(dim=-1).values  # (K,)
    lo, hi = equation.perturbative_gap(EPS0, alpha, omega)
    inside = k_band[growth > 1e-6]
    i_gap = torch.argmin((k - k_gap).abs()).item()
    measured = solve.growth_rate_measured(spectra, times_p, i_gap, t_max / 2)
    print(f"periodic modulation α = {alpha}, Ω = {omega}:  first momentum gap")
    if len(inside):
        first = inside[inside < 1.5 * k_gap]
        print(f"  Floquet:      k ∈ [{first.min().item():.4f}, {first.max().item():.4f}]   "
              f"max growth Im ω = {growth.max().item():.4f}")  # fmt: skip
    print(f"  perturbative: k ∈ [{lo:.4f}, {hi:.4f}]   max growth αΩ/8 = {alpha * omega / 8:.4f}")
    print(f"  simulated growth rate of the mode k = {k[i_gap].item():.4f}: {measured:.4f}")

    return plot.animate(
        x.cpu(), times, D_jump.cpu(), times_p, D_per.cpu(), spectra.cpu(), k.cpu(), k_band.cpu(), w_F.cpu(),
        (lo, hi), (T_f, T_b), (EPS0, eps2, t_jump), (alpha, omega), out_dir / "photonic_time_crystal.gif",
    )  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/photonic_time_crystal/equation.py
r"""Physics → tensor translation: 1D light in a medium whose permittivity changes in time, ε = ε(t).

Maxwell's equations for E_y, H_z (units with c = 1 in vacuum, μ = 1, so B = H):

    ∂_t D = -∂_x H,     ∂_t B = -∂_x E,     D = ε(t) E

The medium is the same everywhere, so every spatial Fourier mode e^{ikx} evolves on its own (k is conserved,
the frequency is not). For the amplitudes u_{ka} = (D_k, B_k), a = 0, 1:

    \dot u_{ka} = A_{kab}(t) u_{kb},     A_k(t) = \begin{pmatrix} 0 & -ik \\ -ik/ε(t) & 0 \end{pmatrix}

(repeated b ⇒ summed). Eliminating B gives \ddot D_k + \frac{k^2}{ε(t)} D_k = 0: a parametric oscillator per k.

Sudden jump ε1 → ε2 (a temporal interface): D and B are continuous (not E and H), the wave splits into a forward
and a backward ("time-reflected") wave at the new frequency ω2 = ω1 n1/n2, with n = \sqrt{ε}.

Periodic ε(t) with period T = 2π/Ω: Floquet theory. The one-period propagator (monodromy matrix) M_k has
eigenvalues λ = e^{-iω_F T}; |λ| > 1 means Im ω_F > 0, exponential growth: a momentum band gap.
"""

import math

import torch


def generator(k: torch.Tensor, eps: float | torch.Tensor) -> torch.Tensor:
    """A_{kab} for every wavenumber: k (K,) → (K, 2, 2) complex."""
    zero = torch.zeros_like(k)
    row0 = torch.stack([zero, -1j * k], dim=-1)  # dD/dt = -ik B
    row1 = torch.stack([-1j * k / eps, zero], dim=-1)  # dB/dt = -ik E = -ik D / ε
    return torch.stack([row0, row1], dim=-2)


def rhs(u: torch.Tensor, k: torch.Tensor, eps: float | torch.Tensor) -> torch.Tensor:
    r"""\dot u_{k...a} = A_{kab} u_{k...b}  (repeated b ⇒ summed). u: (K, 2) or (K, 2, 2) for the monodromy."""
    A = generator(k, eps)
    return torch.einsum("kab,kb...->ka...", A, u)


def periodic_eps(t: float, eps0: float, alpha: float, omega: float) -> float:
    """ε(t) = ε0 (1 + α cos Ωt)."""
    return eps0 * (1.0 + alpha * math.cos(omega * t))


def temporal_fresnel(n1: float, n2: float) -> tuple[float, float]:
    r"""Amplitudes of the forward and backward D waves after a jump n1 → n2, for an incident forward wave D = 1.

    A forward wave has B = D/n, a backward wave B = -D/n. Continuity of D and of B at the jump:
        D_f + D_b = 1,     (D_f - D_b)/n_2 = 1/n_1
    ⇒  D_f = (1 + n_2/n_1)/2,    D_b = (1 - n_2/n_1)/2.
    """
    return 0.5 * (1 + n2 / n1), 0.5 * (1 - n2 / n1)


def split_directions(D: torch.Tensor, B: torch.Tensor, n: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Forward part (B = D/n) and backward part (B = -D/n) of a field in a medium of index n."""
    return 0.5 * (D + n * B), 0.5 * (D - n * B)


def floquet_frequencies(eigenvalues: torch.Tensor, period: float) -> torch.Tensor:
    """λ = e^{-iω T}  ⇒  ω = i ln(λ) / T   (complex; Re ω folded into (-Ω/2, Ω/2], Im ω = growth rate)."""
    return 1j * torch.log(eigenvalues) / period


def perturbative_gap(eps0: float, alpha: float, omega: float) -> tuple[float, float]:
    r"""First momentum gap to first order in α: parametric resonance ω_k = k/n_0 ≈ Ω/2.

    \ddot D + ω_k^2 (1 - α\cos Ωt) D ≈ 0 is unstable for |Ω - 2ω_k| < α ω_k / 2, i.e. k ∈ n_0 Ω/2 (1 ∓ α/4).
    """
    k0 = eps0**0.5 * omega / 2
    return k0 * (1 - alpha / 4), k0 * (1 + alpha / 4)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/photonic_time_crystal/tensors.py
"""Build the concrete tensors: periodic x grid, its wavenumbers, the wave packet and the k list for the bands."""

import math

import torch

BOX = 2 * math.pi * 64  # periodic box length → wavenumber spacing 1/64
N_GRID = 1024
X_START = -30.0  # where the pulse starts
WIDTH = 8.0  # Gaussian envelope width σ


def grid(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """x (X,) on [-BOX/2, BOX/2) and the FFT wavenumbers k (K = X,). float64 for clean Floquet eigenvalues."""
    dx = BOX / N_GRID
    x = (torch.arange(N_GRID, device=device, dtype=torch.float64) - N_GRID // 2) * dx
    k = 2 * math.pi * torch.fft.fftfreq(N_GRID, d=dx).to(device=device, dtype=torch.float64)
    return x, k


def forward_pulse(x: torch.Tensor, k_carrier: float, n: float) -> tuple[torch.Tensor, torch.Tensor]:
    """A right-moving wave packet: D = e^{-(x-x0)²/2σ²} cos(k_c (x - x0)), B = D/n.  Each (X,)."""
    s = x - X_START
    D = torch.exp(-(s**2) / (2 * WIDTH**2)) * torch.cos(k_carrier * s)
    return D, D / n


def band_wavenumbers(k_max: float, n: int, device: torch.device) -> torch.Tensor:
    """The k values for the Floquet band structure (K,), a batch: every k is an independent 2×2 problem."""
    return torch.linspace(0.0, k_max, n + 1, device=device, dtype=torch.float64)[1:]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/photonic_time_crystal/solve.py
"""Spectral time stepping (every Fourier mode k at once, RK4) and the Floquet monodromy matrix, batched over k."""

from typing import Callable

import torch

from problems._common.integrators import rk4_step
from problems._common.progress import steps

from .equation import rhs

EpsOfT = Callable[[float], float]


def evolve(u: torch.Tensor, k: torch.Tensor, eps: EpsOfT, t0: float, dt: float, n_steps: int, every: int = 1):
    """RK4 for \\dot u_{ka} = A_{kab}(t) u_{kb}. Returns the times and the states every `every` steps (incl. start)."""
    times, states = [t0], [u]
    t = t0
    for i in steps(n_steps, "RK4 steps"):
        u = rk4_step(lambda y, s: rhs(y, k, eps(s)), u, t, dt)
        t = t0 + (i + 1) * dt
        if (i + 1) % every == 0:
            times.append(t)
            states.append(u)
    return times, torch.stack(states)


def to_modes(D: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Real fields (X,) → Fourier amplitudes u_{ka} = (D_k, B_k): (K, 2) complex."""
    return torch.stack([torch.fft.fft(D), torch.fft.fft(B)], dim=-1)


def to_field(u: torch.Tensor, component: int = 0) -> torch.Tensor:
    """Fourier amplitudes (..., K, 2) → D(x) (component 0) or B(x) (component 1), shape (..., X)."""
    return torch.fft.ifft(u[..., component], dim=-1).real


def envelope(D: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """Envelope |D_+(x)| of a wave packet from its analytic signal (keep only k > 0, doubled): (X,)."""
    return torch.fft.ifft(2 * torch.fft.fft(D) * (k > 0)).abs()


def interface(D0, B0, k, eps1: float, eps2: float, t_jump: float, t_max: float, dt: float, every: int):
    """A sudden jump ε1 → ε2 at t_jump, integrated in two pieces so no RK4 step straddles the jump.

    D and B (the state u) simply carry over: that *is* the continuity condition at a temporal interface.
    """
    n1 = round(t_jump / dt)
    n2 = round((t_max - t_jump) / dt)
    t_a, u_a = evolve(to_modes(D0, B0), k, lambda t: eps1, 0.0, dt, n1, every)
    t_b, u_b = evolve(u_a[-1], k, lambda t: eps2, t_a[-1], dt, n2, every)
    return torch.tensor(t_a + t_b[1:]), torch.cat([u_a, u_b[1:]])


def monodromy(k: torch.Tensor, eps: EpsOfT, period: float, n_steps: int = 400) -> torch.Tensor:
    """M_{kab}: the propagator over one period for every k, by evolving the 2×2 identity: (K, 2, 2)."""
    identity = torch.eye(2, dtype=torch.complex128, device=k.device).expand(len(k), 2, 2).clone()
    dt = period / n_steps
    M = identity
    for i in range(n_steps):
        M = rk4_step(lambda y, s: rhs(y, k, eps(s)), M, i * dt, dt)
    return M


def growth_rate_measured(spectra: torch.Tensor, times: torch.Tensor, index: int, t_fit: float) -> float:
    """Fit ln|D_k(t)| over the last part (t > t_fit) of the run for one mode: the measured growth rate."""
    keep = times > t_fit
    y = torch.log(spectra[keep, index])
    t = times[keep].to(y.dtype)
    t_c = t - t.mean()
    return (torch.sum(t_c * (y - y.mean())) / torch.sum(t_c**2)).item()


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/photonic_time_crystal/plot.py
"""Animate both experiments: time reflection at a jump of ε, and growth inside the momentum gap, with the bands."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

BLUE, RED, GREY, AMBER, PURPLE = "#2563eb", "#dc2626", "#6b7280", "#f59e0b", "#9333ea"


def animate(
    x: torch.Tensor,
    times: torch.Tensor,
    D_jump: torch.Tensor,
    times_p: torch.Tensor,
    D_per: torch.Tensor,
    spectra: torch.Tensor,
    k: torch.Tensor,
    k_band: torch.Tensor,
    w_F: torch.Tensor,
    gap: tuple[float, float],
    fresnel: tuple[float, float],
    jump: tuple[float, float, float],
    modulation: tuple[float, float],
    path: Path,
) -> Path:
    xs, t = x.numpy(), times.numpy()
    Dj = D_jump.numpy()
    eps1, eps2, t_jump = jump
    alpha, omega = modulation
    lo, hi = gap
    k_gap = 0.5 * (lo + hi)

    fig = plt.figure(figsize=(8.4, 5.4))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.2, 1])
    ax_j, ax_s = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])
    ax_eps, ax_b = fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])

    # (a) temporal interface in real space
    view = np.abs(xs) < 85
    ax_j.set_xlim(-85, 85)
    ax_j.set_ylim(-1.8, 1.8)
    for level, color, name in ((fresnel[0], BLUE, "forward"), (fresnel[1], RED, "backward")):
        ax_j.axhline(abs(level), color=color, ls="--", lw=0.9, alpha=0.7)
        ax_j.text(84, abs(level) + 0.04, f"{name} {level:+.2f}", ha="right", va="bottom", fontsize=7, color=color)
    (line_j,) = ax_j.plot([], [], color="#111827", lw=1.0)
    ax_j.set_xlabel("x", fontsize=9)
    ax_j.set_ylabel("D(x, t)", fontsize=9)
    ax_j.set_title(f"sudden jump ε: {eps1:g} → {eps2:g} at t = {t_jump:g}", fontsize=10)
    label_j = ax_j.text(0.02, 0.04, "", transform=ax_j.transAxes, fontsize=8)
    ax_j.tick_params(labelsize=8)

    # (b) periodic modulation: spectrum |D_k(t)| on a log scale
    spec = spectra.numpy()
    ks = k.numpy()
    show = (ks > 0) & (ks < 2.4 * k_gap)
    order = np.argsort(ks[show])
    k_show = ks[show][order]
    spec_show = spec[:, show][:, order]  # (S, K_show)
    ax_s.axvspan(lo, hi, color="#fde68a", alpha=0.6, lw=0, label="gap (perturbative)")
    ax_s.plot(k_show, np.maximum(spec_show[0], 1e-12), color=GREY, lw=1, label="t = 0")
    (line_s,) = ax_s.plot([], [], color=PURPLE, lw=1.4, label="now")
    ax_s.set_yscale("log")
    ax_s.minorticks_off()
    ax_s.set_ylim(spec_show[0].max() * 1e-3, spec_show.max() * 5)
    ax_s.set_xlim(0, k_show[-1])
    ax_s.set_xlabel("wavenumber k", fontsize=9)
    ax_s.set_ylabel("|D_k(t)|", fontsize=9)
    ax_s.set_title(f"periodic ε = 1 + {alpha:g} cos {omega:g}t", fontsize=10)
    ax_s.legend(loc="upper right", fontsize=7)
    ax_s.tick_params(labelsize=8)

    # (c) the two permittivities ε(t)
    t_fine = np.linspace(0, max(t[-1], times_p[-1].item()), 1500)
    ax_eps.plot(t_fine, np.where(t_fine < t_jump, eps1, eps2), color=BLUE, lw=1.6, label="sudden jump")
    ax_eps.plot(t_fine, eps1 * (1 + alpha * np.cos(omega * t_fine)), color=PURPLE, lw=1.0, label="periodic")
    cursor = ax_eps.axvline(0, color=AMBER, lw=1.5)
    ax_eps.set_xlim(0, t_fine[-1])
    ax_eps.set_xlabel("time t", fontsize=9)
    ax_eps.set_ylabel("ε(t)", fontsize=9)
    ax_eps.legend(loc="center right", fontsize=7)
    ax_eps.tick_params(labelsize=8)

    # (d) Floquet bands: Re ω (folded into [0, Ω/2]) and the growth rate Im ω
    kb = k_band.numpy()
    re = np.abs(w_F.real.numpy())  # (K, 2)
    growth = w_F.imag.numpy().max(axis=-1)
    ax_b.axvspan(lo, hi, color="#fde68a", alpha=0.6, lw=0)
    ax_b.plot(np.repeat(kb, 2), re.reshape(-1), ".", color=BLUE, ms=1.5)
    ax_b.set_ylim(0, omega / 2 * 1.08)
    ax_b.set_xlim(0, kb[-1])
    ax_b.set_xlabel("wavenumber k", fontsize=9)
    ax_b.set_ylabel("Re ω (folded)", fontsize=9, color=BLUE)
    ax_b.set_title("Floquet bands from eig(M_k)", fontsize=10)
    ax_b.tick_params(labelsize=8)
    ax_g = ax_b.twinx()
    ax_g.plot(kb, growth, color=RED, lw=1.4)
    ax_g.set_ylim(0, max(growth.max(), 1e-3) * 1.15)
    ax_g.set_ylabel("growth Im ω", fontsize=9, color=RED)
    ax_g.tick_params(labelsize=8)
    ax_g.grid(False)

    t_p = times_p.numpy()

    def update(s: int):
        line_j.set_data(xs[view], Dj[s, view])
        phase = "before the jump" if t[s] < t_jump else "after: forward + time-reflected wave"
        label_j.set_text(f"t = {t[s]:.0f}   {phase}")
        line_s.set_data(k_show, np.maximum(spec_show[s], 1e-12))
        cursor.set_xdata([t_p[s], t_p[s]])

    fig.tight_layout()
    n_frames = min(len(t), len(t_p))  # the two runs share the time grid
    return save_gif(fig, update, frame_indices(n_frames, 70), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'eps2': 4.0,
    'alpha': 0.2,
    'omega': 2.0,
    't_max': 100.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.photonic_time_crystal")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.photonic_time_crystal.main")
problem_dir = Path("problems/waves/photonic_time_crystal")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
